# Traffic Detection and Tracker PROJECT by Krrish Jain

In [1]:
!pip install -q ultralytics supervision opencv-python-headless

from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
from collections import defaultdict, deque
import time
import os
import json

print(" All imports working!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 All imports working!


In [3]:
! mkdir project

mkdir: cannot create directory ‘project’: File exists


In [4]:
video_path = '/content/project/real_traffic.mp4'

In [5]:
# Check video
cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f" {w}x{h} | {fps}fps | {total} frames")


 768x432 | 29.97002997002997fps | 298 frames


In [6]:
class CameraCalibration:
  """
  converts pixels distances into real word meters
  """

  def __init__(self,pixel_to_meter_ratio=None):
    # Default: guess based on typical highway camera
    # For accurate results, measure a known distance in your video
    self.pixel_to_meter_ratio = pixel_to_meter_ratio or 0.04  # meters per pixel

    # These should be tuned for your camera angle
    self.src_points = None  # Will auto-init from frame size
    self.dst_points = None

  def auto_init_from_frame(self,frame_w,frame_h):
    """
    Auto-initialize perspective transform for typical traffic camera.
    """
    # Source: trapezoid in image (road region)
    self.src_points = np.float32([
        [frame_w * 0.2, frame_h * 0.6],   # top-left
        [frame_w * 0.8, frame_h * 0.6],   # top-right
        [frame_w * 0.9, frame_h * 0.95],  # bottom-right
        [frame_w * 0.1, frame_h * 0.95]   # bottom-left
    ])

    # Destination: rectangle in bird's eye view
    self.dst_points = np.float32([
        [frame_w * 0.2, frame_h * 0.1],
        [frame_w * 0.8, frame_h * 0.1],
        [frame_w * 0.8, frame_h * 0.9],
        [frame_w * 0.2, frame_h * 0.9]
    ])

    self.perspective_matrix = cv2.getPerspectiveTransform(self.src_points, self.dst_points)
    self.inv_matrix = cv2.getPerspectiveTransform(self.dst_points, self.src_points)


  def pixel_to_meter(self, pixel_dist, y_position):
        """
        Convert pixel distance to meters.
        Accounts for perspective (objects further away appear smaller).
        """
        # Simple version: scale by position (closer = larger pixels per meter)
        # At bottom of frame (close): pixels are "bigger"
        # At top of frame (far): pixels are "smaller"
        scale = 1.0 + (1.0 - y_position) * 0.5  # perspective correction
        return pixel_dist * self.pixel_to_meter_ratio * scale

  def transform_point(self, x, y):
        """Transform point to bird's eye view for accurate distance."""
        if self.perspective_matrix is not None:
            pt = cv2.perspectiveTransform(
                np.array([[[x, y]]], dtype=np.float32),
                self.perspective_matrix
            )
            return pt[0][0]
        return x, y
calib = CameraCalibration(pixel_to_meter_ratio=0.035)
calib.auto_init_from_frame(w, h)
print("✅ Camera calibrated!")


✅ Camera calibrated!


In [11]:
class TrafficAnalyzer:
  """
  Production-grade traffic analysis using ByteTrack, speed estimation,
  and comprehensive analytics
  """
  def __init__(self,model,fps=30,calibration=None):
    self.model = model
    self.fps = fps
    self.calibration = calibration or CameraCalibration()

    #ByteTrack tracker (SOTA)
    self.tracker = sv.ByteTrack(
        track_activation_threshold=0.25,
        lost_track_buffer= 30,
        minimum_matching_threshold= 0.8,
        frame_rate= fps,
        minimum_consecutive_frames= 3
        )
    #state
    self.track_history = defaultdict(lambda: deque(maxlen=60))
    self.speed_history = defaultdict(lambda: deque(maxlen=10))
    self.total_count = 0
    self.counted_ids = set()
    self.count_line_y = None

    #Vehicle class mapping
    self.vehicle_classes = {2: 'car',3: 'motorcycle', 5: 'bus', 7: 'truck'}

    #Annotators
    self.box_annotator = sv.BoxAnnotator(thickness=2)
    self.label_annotator = sv.LabelAnnotator(text_thickness=1,text_scale=0.5)
    self.trace_annotater = sv.TraceAnnotator(thickness=2,trace_length=60)

    #Analytics storage
    self.frame_analytics=[]
    self.vehicle_database={}


  def estimate_speed(self, track_id, current_pos, frame_num):
    """
    Estimate speed using Kalman-smoothed trajectory.
    Uses multiple frames for stability.
    """
    history = self.track_history[track_id]
    x, y = current_pos
    history.append((x, y, frame_num))  # Unpack to create (x, y, frame_num) tuple

    if len(history) < 10:  # not enough history
        return None

    # use frames 10 apart for stable estimate
    x1, y1, t1 = history[-10]
    x2, y2, t2 = history[-1]

    # transform to bird eye view for accurate distance
    if self.calibration:
        p1 = self.calibration.transform_point(x1, y1)
        p2 = self.calibration.transform_point(x2, y2)
        dx = p2[0] - p1[0]
        dy = p2[1] - p1[1]
        pixel_dist = np.sqrt(dx**2 + dy**2)
    else:
        pixel_dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    # time in seconds
    time_diff = (t2 - t1) / self.fps

    if time_diff < 0.3:  # we need meaningful time
        return None

    # convert to meters then km/h
    meters = self.calibration.pixel_to_meter(pixel_dist, y2) if self.calibration else pixel_dist * 0.04
    speed_ms = meters / time_diff
    speed_kmh = speed_ms * 3.6

    # sanity check
    if speed_kmh > 200 or speed_kmh < 0:
        return None

    # Smooth with history
    self.speed_history[track_id].append(speed_kmh)
    # use median for more robustness
    smoothed_speed = np.median(self.speed_history[track_id])

    return smoothed_speed

  def detect_direction(self, track_id, current_y):
    """
    Determine if vehicle is approaching or leaving
    """

    history = self.track_history[track_id]
    if len(history) < 3:
      return 'unknown'

    #compare current to average of last 3 positions
    recent_y = np.mean([h[1] for h in list(history)[-3:]])
    prev_y = np.mean([h[1] for h in list(history)[:3]])

    if recent_y > prev_y:
      return 'approaching' # Moving down in frame (closer)
    else:
      return 'leaving' # Moving up in frame (farther)

  def count_vehicle(self,track_id,centre_y,direction):
    """
    Count vehice when they cross the counting line
    """

    if track_id in self.counted_ids or self.count_line_y is None:
      return False

    #count when approaching vehicle cross line going down
    if direction == 'approaching':
        history = self.track_history[track_id]
        if len(history)>=2 :
            prev_y = history[-2][1]
            if prev_y < self.count_line_y and centre_y>= self.count_line_y:
                self.total_count +=1
                self.counted_ids.add(track_id)

                #record in database
                self.vehicle_database[track_id] = {
                              'counted_at_frame': len(self.frame_analytics),
                              'direction': direction
                }
                return True

    return False





  def process_frame(self, frame, frame_num):
    """
    Full pipeline: detect -> track -> analyze -> annotate.
    """
    h, w = frame.shape[:2]
    if self.count_line_y is None:
        self.count_line_y = int(h * 0.6)

    # Detections
    results = self.model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Filter Vehicles
    mask = np.isin(detections.class_id, list(self.vehicle_classes.keys()))
    detections = detections[mask]

    # Handle empty detections
    if len(detections) == 0:
        annotated = frame.copy()
        cv2.line(annotated, (0, self.count_line_y), (w, self.count_line_y), (0, 255, 255), 2)
        analytics = {
            'frame': frame_num,
            'timestamp': frame_num / self.fps,
            'vehicle_count': self.total_count,
            'active_tracks': 0,
            'speeds': {},
            'directions': {}
        }
        self.frame_analytics.append(analytics)
        return annotated, analytics

    # Tracking Vehicles
    detections = self.tracker.update_with_detections(detections)

    # Analytics
    labels = []
    speeds = {}
    directions = {}

    # Correct iteration over sv.Detections
    for i in range(len(detections)):
        bbox = detections.xyxy[i]  # [x1, y1, x2, y2]
        conf = detections.confidence[i]
        cls_id = detections.class_id[i]
        track_id = detections.tracker_id[i]

        x1, y1, x2, y2 = bbox
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

        # Speed estimation
        speed = self.estimate_speed(track_id, (cx, cy), frame_num)
        speeds[track_id] = speed

        # Directions
        direction = self.detect_direction(track_id, cy)
        directions[track_id] = direction

        # Count Vehicle
        self.count_vehicle(track_id, cy, direction)

        # Label
        class_name = self.vehicle_classes.get(cls_id, 'vehicle')
        if speed:
            label = f"#{track_id} {class_name} {speed:.1f}km/h"
        else:
            label = f"#{track_id} {class_name} ..."
        labels.append(label)

    # Annotations
    annotated = frame.copy()

    # Counting Line
    cv2.line(annotated, (0, self.count_line_y), (w, self.count_line_y), (0, 255, 255), 2)
    cv2.putText(annotated, "COUNT LINE", (10, self.count_line_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    # Boxes, Labels, Traces
    annotated = self.box_annotator.annotate(scene=annotated, detections=detections)
    annotated = self.label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
    annotated = self.trace_annotater.annotate(scene=annotated, detections=detections)

    # HUD
    hud_bg = annotated.copy()
    cv2.rectangle(hud_bg, (0, 0), (350, 120), (0, 0, 0), -1)
    annotated = cv2.addWeighted(annotated, 1, hud_bg, 0.3, 0)

    cv2.putText(annotated, f"TOTAL COUNT: {self.total_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(annotated, f"ACTIVE: {len(detections)}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(annotated, f"FPS: {self.fps}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Store Analytics
    analytics = {
        'frame': frame_num,
        'timestamp': frame_num / self.fps,
        'vehicle_count': self.total_count,
        'active_tracks': len(detections),
        'speeds': {int(k): round(v, 2) if v else None for k, v in speeds.items()},
        'directions': {int(k): v for k, v in directions.items()}
    }

    self.frame_analytics.append(analytics)

    return annotated, analytics



In [12]:
# Initialize
model = YOLO("yolov8s.pt")
analyzer = TrafficAnalyzer(model, fps=fps, calibration=calib)

# Open video
cap = cv2.VideoCapture(video_path)
output_path = "/content/output_tracked.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

print(f" Processing {total} frames...")
start_time = time.time()

for frame_num in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    annotated, _ = analyzer.process_frame(frame, frame_num)
    out.write(annotated)

    if frame_num % 30 == 0:
        elapsed = time.time() - start_time
        speed = (frame_num + 1) / elapsed
        print(f"Frame {frame_num}/{total} | {speed:.1f} FPS | Count: {analyzer.total_count}")

cap.release()
out.release()

elapsed = time.time() - start_time
print(f"\n{'='*50}")
print(f"DONE")
print(f"   Frames: {total}")
print(f"   Time: {elapsed:.1f}s")
print(f"   Speed: {total/elapsed:.1f} FPS")
print(f"   Vehicles counted: {analyzer.total_count}")
print(f"   Unique tracks: {len(analyzer.vehicle_database)}")
print(f"{'='*50}")



 Processing 298 frames...
Frame 0/298 | 5.5 FPS | Count: 0
Frame 30/298 | 28.7 FPS | Count: 1
Frame 60/298 | 30.6 FPS | Count: 2
Frame 90/298 | 31.4 FPS | Count: 5
Frame 120/298 | 31.8 FPS | Count: 9
Frame 150/298 | 32.0 FPS | Count: 11
Frame 180/298 | 32.5 FPS | Count: 13
Frame 210/298 | 33.0 FPS | Count: 15
Frame 240/298 | 33.3 FPS | Count: 19
Frame 270/298 | 32.4 FPS | Count: 20

DONE
   Frames: 298
   Time: 9.4s
   Speed: 31.7 FPS
   Vehicles counted: 21
   Unique tracks: 21


In [19]:
from google.colab import files
files.download('/content/output_tracked.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>